In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
category_df = pd.read_csv(r"C:\lecture\3rd_project\database\sql\source_csv\category.csv")
category_df.head()

,category_code,category_name,description
0,CAT0000,고깃집,신선한 육류를 불판에 직접 구워 먹는 식당. 회식 가족 모임 든든한 저녁 식사에 적...
1,CAT0001,짜장면,달콤짭짤한 춘장 소스에 면을 비벼 먹는 대중적인 중화요리. 호불호가 없으며 배달로 ...
2,CAT0002,일식당,정갈하고 깔끔한 일본식 요리와 코스를 제공하는 식당. 손님 대접이나 격식 있는 자리...
3,CAT0003,숙성회,일정 시간 숙성하여 감칠맛과 차진 식감을 극대화한 생선회. 사케나 전통주와 잘 어울...
4,CAT0004,돼지갈비,달콤하고 짭조름한 간장 양념이 밴 돼지고기 구이. 남녀노소 호불호가 적어 외식이나 ...


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
import numpy as np
import base64


# 모델 초기화
llm = init_chat_model('gpt-4o-mini', model_provider="openai", temperature=0)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 텍스트 합치기 및 임베딩 진행
category_df['text_to_embed'] = category_df['category_name'] + " : " + category_df['description']
embeddings = embedding_model.embed_documents(category_df['text_to_embed'].tolist())

# Base64 인코딩 함수 만들기
def encode_to_base64_blob(emb_vector):
    vec = np.array(emb_vector, dtype=np.float32) 
    blob = vec.tobytes()                        
    return base64.b64encode(blob).decode('utf-8') 

# 데이터프레임의 새로운 컬럼에 인코딩된 데이터 저장
category_df['embedding'] = [encode_to_base64_blob(emb) for emb in embeddings]

# --- (참고) 나중에 CSV에서 읽어와서 다시 복원할 때 쓸 함수 ---
def decode_from_base64_blob(encoded_str):
    blob = base64.b64decode(encoded_str)
    return np.frombuffer(blob, dtype=np.float32)

# 5. 최종 결과 확인
category_df.head()

c:\Users\Playdata\miniconda3\envs\project_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,category_code,category_name,description,text_to_embed,embedding
0,CAT0000,고깃집,신선한 육류를 불판에 직접 구워 먹는 식당. 회식 가족 모임 든든한 저녁 식사에 적...,고깃집 : 신선한 육류를 불판에 직접 구워 먹는 식당. 회식 가족 모임 든든한 저녁...,ACAePADAiz0AQDK9AKD4vAAg/joAgJe8AACINQBADzwAoN...
1,CAT0001,짜장면,달콤짭짤한 춘장 소스에 면을 비벼 먹는 대중적인 중화요리. 호불호가 없으며 배달로 ...,짜장면 : 달콤짭짤한 춘장 소스에 면을 비벼 먹는 대중적인 중화요리. 호불호가 없으...,AGBFugBAijwA4Cy9ACCEuwCgujwAoG28AOALPQBgCD0AQB...
2,CAT0002,일식당,정갈하고 깔끔한 일본식 요리와 코스를 제공하는 식당. 손님 대접이나 격식 있는 자리...,일식당 : 정갈하고 깔끔한 일본식 요리와 코스를 제공하는 식당. 손님 대접이나 격식...,AGAevAAARrsAAGS9AOAdvABAxD0AAK48AKCBvQAgez0AgH...
3,CAT0003,숙성회,일정 시간 숙성하여 감칠맛과 차진 식감을 극대화한 생선회. 사케나 전통주와 잘 어울...,숙성회 : 일정 시간 숙성하여 감칠맛과 차진 식감을 극대화한 생선회. 사케나 전통주...,AECOPADAOrsAQIS8AKA8vQAAhrwAQPy8AOApvADA8TwA4G...
4,CAT0004,돼지갈비,달콤하고 짭조름한 간장 양념이 밴 돼지고기 구이. 남녀노소 호불호가 적어 외식이나 ...,돼지갈비 : 달콤하고 짭조름한 간장 양념이 밴 돼지고기 구이. 남녀노소 호불호가 적...,AOAAPQBACTwAwHC9AABIPQAApTwA4J07AIBIOwBAITsAwD...


In [ ]:
import pandas as pd

# 1. 임베딩 및 Base64 인코딩이 완료된 데이터프레임을 다시 CSV로 저장
# 파일명을 동일하게 'category.csv'로 하면 덮어쓰기가 됩니다!
category_df.to_csv('category.csv', index=False, encoding='utf-8-sig')

print("✅ category.csv 파일에 저장완료")

# 2. 파일이 잘 저장되었는지 다시 불러와서 head() 확인하기
saved_df = pd.read_csv('category.csv')
saved_df.head()

✅ category.csv 파일에 성공적으로 덮어쓰기 되었습니다!


,category_code,category_name,description,text_to_embed,embedding,similarity,embedding_blob
0,CAT0000,고깃집,신선한 육류를 불판에 직접 구워 먹는 식당. 회식 가족 모임 든든한 저녁 식사에 적...,고깃집 : 신선한 육류를 불판에 직접 구워 먹는 식당. 회식 가족 모임 든든한 저녁...,ACAePADAiz0AQDK9AKD4vAAg/joAgJe8AACINQBADzwAoN...,0.366098,b'\x00 \x1e<\x00\xc0\x8b=\x00@2\xbd\x00\xa0\xf...
1,CAT0001,짜장면,달콤짭짤한 춘장 소스에 면을 비벼 먹는 대중적인 중화요리. 호불호가 없으며 배달로 ...,짜장면 : 달콤짭짤한 춘장 소스에 면을 비벼 먹는 대중적인 중화요리. 호불호가 없으...,AGBFugBAijwA4Cy9ACCEuwCgujwAoG28AOALPQBgCD0AQB...,0.361832,"b'\x00`E\xba\x00@\x8a<\x00\xe0,\xbd\x00 \x84\x..."
2,CAT0002,일식당,정갈하고 깔끔한 일본식 요리와 코스를 제공하는 식당. 손님 대접이나 격식 있는 자리...,일식당 : 정갈하고 깔끔한 일본식 요리와 코스를 제공하는 식당. 손님 대접이나 격식...,AGAevAAARrsAAGS9AOAdvABAxD0AAK48AKCBvQAgez0AgH...,0.248674,b'\x00`\x1e\xbc\x00\x00F\xbb\x00\x00d\xbd\x00\...
3,CAT0003,숙성회,일정 시간 숙성하여 감칠맛과 차진 식감을 극대화한 생선회. 사케나 전통주와 잘 어울...,숙성회 : 일정 시간 숙성하여 감칠맛과 차진 식감을 극대화한 생선회. 사케나 전통주...,AECOPADAOrsAQIS8AKA8vQAAhrwAQPy8AOApvADA8TwA4G...,0.301480,b'\x00@\x8e<\x00\xc0:\xbb\x00@\x84\xbc\x00\xa0...
4,CAT0004,돼지갈비,달콤하고 짭조름한 간장 양념이 밴 돼지고기 구이. 남녀노소 호불호가 적어 외식이나 ...,돼지갈비 : 달콤하고 짭조름한 간장 양념이 밴 돼지고기 구이. 남녀노소 호불호가 적...,AOAAPQBACTwAwHC9AABIPQAApTwA4J07AIBIOwBAITsAwD...,0.277355,b'\x00\xe0\x00=\x00@\t<\x00\xc0p\xbd\x00\x00H=...
